## **1. Setup and Imports**

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display setup
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", '{:.2f}'.format)

# plot setup

%matplotlib inline

###  **1.1 Load data**:

In [3]:
df = sns.load_dataset(("titanic"))

## **2. Data description**

### **2.1. Dataset**
This is the Titanic dataset with 15 variables and 891 instances. The dataset contains demographic information of passengers on a shipwreck, including their age, gender, the port of embarkation, the class of their deck, and whether or not they survived. Each row represent individual passengers on board.

### **2.2 Inspecting the dataset**

In [4]:
df.info()
print(df.shape) # total (rows, cols)
print(df.size) # total cells
print(df.describe())#df.describe(include = "all")
print(df.head()) # first 6 rows
print(df.dtypes) # type of cols
#add to see if it changes
print(df.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB
(891, 15)
13365
       survived  pclass    age  sibsp  parch   fare
count    89

## **3. Missing values**
### **3.1. Finding missing values**
Out of the 15 variables, only 4 columns contain missing values, with the 'deck' column having nearly 77% missing values, followed by the 'age' variable. The 'embarked' and 'embarked_town' variables each have only about 1% missing values (likely corresponding to the same data points).

In [ ]:
#finding missings values, isnull() only returns a boolean value where the values is missing. include sum() to get the total number of missing values per variable
df.isnull().sum()
#df.isnull().any() #this will return a boolean value of True if there is at least one missing value in the dataset, and False if there are no missing values in the dataset

In [ ]:
#list of columns that have missing values in the dataset
print([col for col in df.columns if df[col].isnull().any()]) 

# total count of missing values in the dataset
total_missing_values = df.isnull().values.sum()
print(total_missing_values)

# Visualize missing values using a heatmap
plt.figure (figsize=(12, 6))
sns.heatmap(df.isnull(), cbar = False, cmap = "viridis")
plt.title("Missing values heatmap", fontsize =20, fontweight = 'bold')
plt.show()


### **3.2. Filling missing values**
* **'age'** column is numeric: to be filled with median or mean.
* **'deck'** column is categorical: to be removed (77% missing values).
* **'embarked'** or **'embark_town'** column is categorical: to be filled with mode.

#### *3.2.1. Age column*

In [ ]:
#this create a boolean series (T or F) if an instance is missing within age column
bool_series = pd.isnull(df["age"])
print(bool_series)

#this returns data for only instances where age is missing
#missing_age = df[pd.isnull(df["age"])]
missing_age = df[bool_series]
print(missing_age)

In [ ]:
df.isna() #returns a dataframe of the same shape as df, indicating missing values with True and non-missing values with False same as df.isnull()

non_missing_age = df[~bool_series] #this returns data for only instances where age is not missing
print(non_missing_age)

#### *Types of filling methods*
##### *1. Specific value*

In [ ]:
# dropna() function is used to remove rows/columns with missing values but can lose a potential information in a dataset. Best approach is use imputation methods like below.
df_fill_age= df.copy()
df_fill_age["age"] = df_fill_age["age"].fillna(0) #this will replace all missing values in the age column with 0
print(df_fill_age["age"])


##### *2. Mean/Median/Mode*

In [ ]:

mean_age = df["age"].mean() #calculate the mean age
median_age = df["age"].median() #calculate the median age
print(f"\nMean of Age:{mean_age}")
print(f"\nMedian of Age: {median_age}")

#mean imputation age column
df_age_mean_imputed = df.copy()
df_age_mean_imputed["age"] = df_age_mean_imputed["age"].fillna(mean_age.round()) #this will replace all missing values in the age column with the mean age
#print(df_age_mean_imputed)

#median imputation age column
df_age_median_imputed = df.copy()
df_age_median_imputed["age"] = df_age_median_imputed["age"].fillna(median_age.round()) #this will replace all missing values in the age column with median age
#print(df_age_median_imputed)

#### *3.2.1. Deck column*

In [ ]:
# mode imputation
## the deck variable has almost 77% missing values, so the variable is deemed to be not useful for analysis, but if we want to impute the missing values with the most common value, we can use the mode imputation method as below.
df_age_median_imputed["deck"].unique() # deck has 7 unique values, but the most common value is C
df_deck_imputed = df_age_median_imputed.copy()
df_deck_imputed["deck"] = df_deck_imputed["deck"].fillna("C") #this will replace all missing values in the deck column with the most common value C
print(df_deck_imputed)

## Remove deck variable completely
df_deck_removed = df_deck_imputed.drop(columns = "deck") #this will remove the deck variable from the dataset

#### *3.2.3. Embarked or Embark_town column*

In [ ]:
# embark_town and embarked variable has 2% missing values, so embark_town will is imputed with the most common value and embarked variable will be removed completely because it has the same information as embark_town variable
df_embark_town_imputed = df_deck_removed.copy()
df_embark_town_imputed["embark_town"].unique() #embark_town variable has 3 unique values, with S being the most common value
df_embark_town_imputed["embark_town"] = df_embark_town_imputed["embark_town"].fillna("Southampton") #this will replace all missing values in the embark_town column with the most common value S

# Remove embarked variable completely
df_embarked_removed = df_embark_town_imputed.drop(columns = "embarked") #this will remove the embarked variable from the dataset

## **Full audit on dataset**

In [ ]:
def audit(df):
    return pd.DataFrame({
        'dtype'     : df.dtypes,
        'nulls'     : df.isnull().sum(),
        'null_%'    : (df.isnull().mean()*100).round(2),
        'unique'    : df.nunique(),
        'unique_%'  : (df.nunique()/len(df) * 100).round(2),
        'zeros'     : (df == 0).sum(),
        'sample'    : df.iloc[0]

    })

audit(df)

### **Cleaned Dataset**

In [ ]:

titanic_data = df_embarked_removed
titanic_data.info()


## **4. Data Visualization**

### **4.1. Univariate analysis**

In [ ]:
#subplot of continuous variables

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows = 2, ncols = 2, figsize = (12,6), gridspec_kw={'height_ratios': [4, 1]})
fig.supylabel("Number of Passengers")
fig.subplots_adjust(left=0.08) 

# age distirbution hisplot
sns.histplot(data = titanic_data, x= "age", edgecolor = 'none', color = '#a1c9f4', ax = ax1)
ax1.set_title("Distribution of Age", fontweight = 'bold')
ax1.set_xlabel("")
ax1.set_ylabel("")
#age distribution boxplot
sns.boxplot(data = titanic_data, x = "age",  color = '#a1c9f4', ax =ax3)
ax3.set_xlabel("Age")


# fare distribution histplot
sns.histplot(data = titanic_data, x= "fare", edgecolor = 'none',stat = "count", bins = 20, color = '#a1c9f4', ax = ax2)
ax2.set_title("Distribution of Ticket Price", fontweight = 'bold')
ax2.set_xlabel("")
ax2.set_ylabel("")
#fare distribution boxplot
sns.boxplot(data = titanic_data, x = "fare", color = '#a1c9f4', ax = ax4)
ax4.set_xlabel("Ticket Fare in Dollars")



#### **4.1.1. Distribution of continuous variables (Age and Fare)**
From the age category, it can be seen that age group between 20 to 30 were the highest on board peaking at 30-35. After 50 years, the number of passengers on board started to show a decrease. The elderly group(60+)were few on board. With an outlier of a passenger around 80 year old. The insignts suggest that alduts between 20 to 35 are free to explore comparing them with individuals above 40 years

Alot of passengers purchased a ticket less than 100 Dollars while only few passengers purchased premium tickes. with only 1 passenger purchased a ticket of over 500 dollars.

In [ ]:
# family size story

fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(nrows=2, ncols=3, figsize=(16,10))
fig.suptitle("Univariate Analysis of Categorical Variables", fontsize = 18, fontweight = 'bold')
fig.supylabel("Number of Passengers")
fig.subplots_adjust(left = 0.06)
total = len(titanic_data)


# survived or not variable
sns.countplot(data = titanic_data, x = "survived", color = '#a1c9f4', ax = ax1)
ax1.set_title("Distribution of Passenger Survival: Yes vs No", fontweight ='bold')
ax1.set_xlabel("")
ax1.set_ylabel("")
ax1.set_xticks(ticks = (0,1), labels = ('No','Yes'))
# add percentages labels to the bars
for container in ax1.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax1.bar_label(container, labels=labels, fontweight ='bold')

# sex variable
sns.countplot( data = titanic_data, x = "sex", color = '#a1c9f4', ax = ax2)
ax2.set_title("Distribution of Passenger sex", fontweight = 'bold')
ax2.set_xlabel("")
ax2.set_ylabel("")
ax2.set_xticks(ticks = ('male', 'female'), labels = ('Male', 'Female'))
# add percentages labels to the bars
for container in ax2.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax2.bar_label(container, labels=labels, fontweight ='bold')


# sex variable
sns.countplot( data = titanic_data, x = "alone", color = '#a1c9f4', ax = ax3)
ax3.set_title("Distribution of Passenger: Alone vs With Family", fontweight ='bold')
ax3.set_xlabel("")
ax3.set_ylabel("")
ax3.set_xticks(ticks = (False, True),labels = ('With Family', 'Alone'))
# add percentages labels to the bars
for container in ax3.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax3.bar_label(container, labels=labels, fontweight ='bold')

# class variable
sns.countplot( data = titanic_data, x = "class", color = '#a1c9f4', ax = ax4)
ax4.set_title("Distribution of Passenger's Class", fontweight = 'bold')
ax4.set_xlabel("")
ax4.set_ylabel("")
ax4.set_xticks(ticks = ('First', 'Second', 'Third'),labels = ('First Class', 'Second Class', 'Third Class'))
# add percentages labels to the bars
for container in ax4.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax4.bar_label(container, labels=labels, fontweight ='bold')

# who variable
sns.countplot( data = titanic_data, x = "who", color = '#a1c9f4', ax = ax5)
ax5.set_title('Distribution of Passengers by Category', fontweight = 'bold')
ax5.set_xlabel("")
ax5.set_ylabel("")
ax5.set_xticks(ticks = ('man', 'woman', 'child'),labels = ('Men', 'Women', 'Children'))



# who variable
sns.countplot( data = titanic_data, x = "embark_town", color = '#a1c9f4', ax = ax6)
ax6.set_title('Distribution of Embarkation Town', fontweight = 'bold')
ax6.set_xlabel("")
ax6.set_ylabel("")
# add percentages labels to the bars
for container in ax6.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax6.bar_label(container, labels=labels, fontweight ='bold')



#### **4.1.2 Distribution of Categorical variables**
The distribution shows that only 38.4% of passengers on board survived. The sex variable also shows that there were lot of males compared to females on board. A number of passengers were on board alone and only 40% of them were with family. Over half (55%) of passengers were on third class, followed by 24% on first class. There were also children on board only accounting for 9.3% of the total passengers. 72.5% of passengers departed from Souhampton followed by 19% from Cherbourg.

### **4.2. Bivariate Analysis**
To understand the rate of survival, it is essential to dive deep into understanding the relationship between the survival rate and the class the passengers they were on, the fare price of their ticket, the town of embarkation, if they had a family on board, their age and gender. 

#### **4.2.1 Family size story**

In [ ]:
# bivariate analysis: family size story
fig,((ax1,ax2), (ax3, ax4)) = plt.subplots(nrows = 2, ncols = 2, figsize = (16,14))
fig.suptitle('Bivariate Analysis of Categorical VariableS', fontsize = 18, fontweight ='bold')
fig.supylabel('Number of Passengers')
fig.subplots_adjust(left =0.06)

# alone? vs survival
sns.countplot(data = titanic_data, x = "alone", hue = "survived", palette = "pastel", ax = ax1)
ax1.set_title("Distribution of Survival Rate: With Family vs Alone", fontweight = 'bold')
ax1.set_xlabel("")
ax1.set_ylabel("")
ax1.set_xticks( ticks = (False, True), labels =("With family", "Alone"))
ax1.legend(labels = ("No", "Yes"), title = "Survived?")
for container in ax1.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax1.bar_label(container, labels=labels, fontweight ='bold')

# family size vs survival rate
# create a family size variable from: siblings or spouse and parent or child
titanic_data["family_size"] = titanic_data["parch"] + titanic_data["sibsp"] +1
sns.countplot(data =titanic_data, x = "family_size", hue = "survived", palette = "pastel", ax = ax2)  
ax2.set_title("Distribution of Survival Rate by Family Size", fontweight = 'bold')
ax2.set_xlabel("")
ax2.set_ylabel("")
ax2.legend(labels = ("No", "Yes"), title = "Survived?")
for container in ax2.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax2.bar_label(container, labels=labels, fontweight ='bold')

#family size variable vs clas
sns.countplot(data =titanic_data, x = "family_size", hue = "class", palette = "pastel", ax = ax3)  
ax3.set_title("Distribution of Class by Family Size", fontweight = 'bold')
ax3.set_xlabel("")
ax3.set_ylabel("")
ax3.legend(title = "Class")
for container in ax3.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax3.bar_label(container, labels=labels, fontweight = 'bold')

#survival by class
sns.countplot(data = titanic_data, x = "class", hue= "survived", palette = "pastel", ax = ax4)
ax4.set_xlabel("")
ax4.set_ylabel("")
ax4.set_title("Distribution of Surival rate by class", fontweight = 'bold')
ax4.legend(labels = ["No", "Yes"], title = "Survived?")
# add percentages labels to the bars
for container in ax4.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax4.bar_label(container, labels=labels, fontweight ='bold')


* Passengers travelling alone had lower chances of survival. Family size between 2 to 4 had better odds of survival, and a drop of rate of survival is seen for larger family group(5+). 
* Higher number of passegers travelling alone is seen to have been on a third class, same trend can be noted for passengers travelling as a family of 5 or more. 
* A family of 2 has the highest number of people in first class, this could indicate young or elderly couples without children and they are better off economically comparing them with larger family group being on third class.
* This can also be linked to survival based on the class passengers were on. The passengers on third class had a lower chance of survival and it was noted that these are mostly people travelling alone or with larger family. Passengers on first class had a better chance of survival maybe they were given priority during rescue.

#### **4.2.2 Port of Embarkation**

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(nrows =1, ncols = 3, figsize = (14,7))
fig.suptitle("The story of survival based on Port of Embarkation", fontsize =12, fontweight = 'bold')
fig.supylabel("Number of passengers")
fig.subplots_adjust(left =0.08)

# survival by town of embarkation
sns.countplot( data = titanic_data, x = "embark_town", hue = "survived", palette = "pastel", ax = ax1)
ax1.set_xlabel("")
ax1.set_ylabel( "")
ax1.set_title("Distribution of Surival rate\nby Town of Embarkation", fontsize = 10, fontweight = 'bold')
ax1.legend(labels = ["No", "Yes"], title = "Survived?")
for container in ax1.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax1.bar_label(container, labels=labels, fontweight ='bold')


# class by town of embarkation
sns.countplot( data = titanic_data, x = "embark_town", hue = "class", palette = "pastel", ax = ax2)
ax2.set_xlabel("")
ax2.set_ylabel( "")
ax2.set_title("Distribution of Class by\nTown of Embarkation", fontsize = 10, fontweight = 'bold')
ax2.legend(title = "Class")
for container in ax2.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax2.bar_label(container, labels=labels, fontweight ='bold')

#survival by class
sns.countplot(data = titanic_data, x = "class", hue= "survived", palette = "pastel", ax = ax3)
ax3.set_xlabel("")
ax3.set_ylabel("")
ax3.set_title("Distribution of Surival rate\nby class", fontsize = 10, fontweight = 'bold')
ax3.legend(labels = ["No", "Yes"], title = "Survived?")
# add percentages labels to the bars
for container in ax3.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax3.bar_label(container, labels=labels, fontweight ='bold')



* 73% of passengers embarked from Southampton followed by 19% from Cherbourg. 
* 39% of passengers who embarked from Southampton were on the third class. Alot of passengers who boarded from Queesstown were on third class. 
* This trend is also complemented by the survival rate based on class passengers they were on, Passengers on third class had a lower chance of survival(39% from Southampton and 8.1% from Queentown and 7.4% from Cherbourg). Passengers of first class had higher chance of survival and most of them embarked from Cherboug. 
* The insights suggest that class played a big role on whether the passenger survives or not. Another take is that the port of embarkation could suggest that Southampton is a lower income nerbourhood compared to higher income Cherbourg.

#### 4.2.3 Social and Economic Layer

In [ ]:
fig, ((ax1, ax2),(ax3, ax4)) = plt.subplots(nrows =2, ncols = 2, figsize = (16,12))
fig.suptitle("The story of survival based on Social and Economic Layer", fontsize =12, fontweight = 'bold')
fig.supylabel("Number of passengers")
fig.subplots_adjust(left =0.08)

# survival by gender
sns.countplot( data = titanic_data, x = "who", hue = "survived", palette = "pastel", ax = ax1)
ax1.set_xlabel("")
ax1.set_ylabel( "")
ax1.set_title("Distribution of Surival Rate by Demographic Group", fontweight = 'bold')
ax1.legend(labels = ["No", "Yes"], title = "Survived?")
# add percentages labels to the bars
for container in ax1.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax1.bar_label(container, labels=labels, fontweight ='bold')

# survival by sex
sns.countplot( data = titanic_data, x = "sex", hue = "survived", palette = "pastel", ax = ax2)
ax2.set_xlabel("")
ax2.set_ylabel( "")
ax2.set_title("Distribution of Surival rate by Sex", fontweight = 'bold')
ax2.legend(labels = ["No", "Yes"], title = "Survived?")
for container in ax2.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax2.bar_label(container, labels=labels, fontweight ='bold')

# survival by the ticket price
# create a new variable with price range
print(f"min_fare: {titanic_data['fare'].min()}")
print(f"max_fare: {titanic_data['fare'].max()})")
print(f"Average_fare: {titanic_data['fare'].mean()}")
#create a new variable based on fare variable
titanic_data['fare_category'] = pd.cut(
    x = titanic_data['fare'],
    bins = [0, 15, 35, 100, float('inf')],
    labels = ["Low Fare", "Medium Fare", "High Fare", "Very High"],
)

sns.countplot(data = titanic_data, x = "fare_category", hue = "survived", palette = "pastel", ax = ax3)
ax3.set_xlabel("")
ax3.set_ylabel( "")
ax3.set_title("Distribution of Surival rate by Ticket Fare Category", fontweight = 'bold')
ax3.legend(labels = ["No", "Yes"], title = "Survived?")
# add percentages labels to the bars
for container in ax3.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax3.bar_label(container, labels=labels, fontweight ='bold')


# age group vs survival
print(f"min age: {titanic_data['age'].min()}")
print(f"avarage age: {titanic_data['age']. mean()}")
print(f" max age: {titanic_data['age'].max()}")

titanic_data["age_category"] = pd.cut(
    x = titanic_data['age'],
    bins = [0, 17, 24, 59, float('inf')],
    labels = ["Child", "Young Adult", "Adult", "Elderly"],
    right = True                                   
)

sns.countplot( data = titanic_data, x = "age_category", hue = "survived", palette = "pastel", ax = ax4)
ax4.set_xlabel("")
ax4.set_ylabel( "")
ax4.set_title("Distribution of Surival rate by Age group", fontweight = 'bold')
ax4.legend(labels = ["No", "Yes"], title = "Survived?")
for container in ax4.containers:
    labels = [f'{100*v/total:.1f}%' for v in container.datavalues]
    ax4.bar_label(container, labels=labels, fontweight ='bold')


* The results highlight that women and children had higher chances of survival over men. This could suggest that women and children were given higher priority.
* On the ticket fare, high and very high tickets shows that chances of survival were high suggesting that passengers were not just buying comfort but also close proximity to lifekackets. Passengers on low fare had higher rate of survival suggesting that they may had to fight to survive since this category had highest number of passengers on board.
* Young adults and adults had a higher count of passengers who did not survive. while the child category shows that 6.8% survived and 5.8% did not survives. This behaviour is expected because children were probably given higher priority compared to adults.


#### **4.2.4. Economic layer vs Sex of Passengers

In [ ]:

g = sns.FacetGrid(titanic_data, col="who", height=4, aspect=0.8)
g.map_dataframe(sns.countplot, x="class", hue="survived", palette="pastel")
g.add_legend(labels = ["No", "Yes"], title="Survived?")
g.set_axis_labels("Fare Category", "Passenger Count")
g.set_titles(col_template="{col_name}")
for ax in g.axes.flat:
    for container in ax.containers:
        labels = [f'{100*v/total:.1f}%' if v > 0 else '' for v in container.datavalues]
        ax.bar_label(container, labels=labels, fontweight='bold', padding=3)



The insights shows that women and children in second and and first class were almost entirely saved, while those in third class had 50/50 chance of survival. The Wealth did not protect men, the "women and children first" rule overrode economic privilege entirely for males.

### **4.3. Multivariate **

In [ ]:
# Pair plot for selected variables
sns.pairplot(titanic_data, vars=['age', 'fare', 'pclass'], hue='survived', palette='pastel')
plt.suptitle('Pair Plot of Age, Fare, and Pclass', y=1.02)
plt.show()